# uniko — a complete memory-API walkthrough (Python)

[`uniko`](https://github.com/rustic-ai/uniko) is an embedded, Rust-native **cognitive memory engine** for
AI agents. It links into your process like SQLite — graph + vector + full-text + logic in one in-process
engine — ingests messages with a **local NLP cascade ($0, no LLM)**, and serves recall with **no LLM in the
hot path**.

This notebook drives the **async-first Python SDK** end-to-end through a single storyline: an assistant
building memory about *Alice* and her project, then recalling, querying, reasoning over, scoping, and
forgetting it.

## Run it

```bash
cd bindings/uniko-py
uv run maturin develop                       # build the extension into the uv venv
uv run --group notebook jupyter lab          # launch this notebook in that venv
```

`maturin` needs `protoc` and a C/C++ toolchain on `PATH` (the stack statically links ONNX Runtime).

> **First-run note.** The first `observe()` / `recall()` downloads ~850 MB of ONNX models
> (BGE-small embeddings + kniv-deberta NLP + a reranker) to `~/.uni_cache/`. The first call is slow;
> every call after is fast. The graph-only sections (query, goals, logic) need no models.

> **Async note.** Cells use top-level `await`, which Jupyter/ipykernel supports natively. Every verb also
> has a blocking `*_sync` twin (shown at the end) for scripts and event-loop-free contexts.


## 1. Open an instance

`Uniko` is the single entry point. `in_memory()` gives an ephemeral instance (great for demos); use
`open("./path")` for a persistent one. `config()` returns the effective settings as a dict.


In [ ]:
import datetime
import uniko

print("uniko version:", uniko.__version__)

uni = await uniko.Uniko.in_memory()
uni.config()

## 2. Agent and session handles

Everything hangs off the handle: `uni.agent(id)` returns an **agent** (a participant that recalls and
reasons), and `agent.session(id)` opens a **conversation**. Reach for the builder when you want to tune
capabilities:

```python
uni = await uniko.Uniko.builder().in_memory().streaming(False).build()
```


In [ ]:
agent = uni.agent("assistant")
session = agent.session("conversation-1")
print("agent:", agent.agent_id, "| session:", session.session_id)

## 3. Observe — write memory

`session.observe(turn)` ingests one message: a local NER + NLP cascade extracts entities and observations,
then **one atomic transaction** writes the Message, Entities, Observations, edges, and chunks — and commits
before returning (idempotent on the turn id). `Turn` is a builder: `.id()`, `.addressed_to()`,
`.metadata()`, `.at()`, `.attach()`.

> The first cell triggers the one-time model download described above.


In [ ]:
conversation = [
    uniko.Turn("alice", "Hi! I'm Alice. I'm building a recall benchmark for my agent.").id("m1"),
    uniko.Turn("assistant", "Nice to meet you, Alice. What stack are you using?")
        .id("m2").addressed_to(["alice"]),
    uniko.Turn("alice", "Rust mostly, and I store all my conversations in uniko.")
        .id("m3").metadata("topic", "stack"),
    uniko.Turn("assistant", "Great choice. I'll remember you use Rust and uniko.")
        .id("m4").addressed_to(["alice"]),
    uniko.Turn("alice", "My deadline for the benchmark is the end of June.")
        .id("m5").metadata("topic", "planning"),
]

for turn in conversation:
    result = await session.observe(turn)
    print(repr(result))

Each `observe()` returns an `ObserveResult` — the node ids it wrote plus what extraction pulled out:


In [ ]:
print("message node:", result.message_node_id)
print("chunks:", result.chunk_node_ids)
print("entities:", result.extracted_entities)        # [(node_id, name), ...]
print("observations:", result.extracted_observations) # [node_id, ...]
print("attachments:", result.attachment_count)

## 4. Recall — read memory (the core)

`agent.recall(query)` runs a coverage-gated cascade — compiled Facts/Procedures/Topics first, then hybrid
vector + BM25 over Episodes/Observations/Messages, then a raw Chunk/Artifact fallback — assembled under a
token budget. **No LLM runs in this path.** Each item carries its `kind`, a fused `score`, and the
`sources` it traces back to.

> **Honest caveat.** Consolidated **Facts** form asynchronously once enough observations accumulate
> (threshold/timer-gated). In a short session like this one, recall mostly surfaces Messages, Observations,
> and Chunks — still with full provenance — rather than compiled Facts.


In [ ]:
bundle = await agent.recall("What do we know about Alice's project?")
print(f"coverage={bundle.coverage:.3f}  total_tokens={bundle.total_tokens}  items={len(bundle)}")
print(f"phase1_only={bundle.phase1_only}  phase2_only={bundle.phase2_only}")
print()

if not bundle.items:
    print("(no items — see the consolidation note above)")
for item in bundle.items:
    print(f"[{item.kind}] score={item.score:.3f}  {item.content[:90]!r}")
    for src in item.sources:
        print(f"     <- {src.kind}: message_id={src.message_id} chunk_id={src.chunk_id}")

## 5. Grounded answers with `answer()` (not run here)

`agent.answer(question)` layers an LLM over `recall()` — it retrieves the same context, then asks a
configured model to synthesize an answer **with citations back to the sources**. It needs an `LlmSpec`
registered on the builder; calling it without one raises `uniko.ConfigError`. We keep this notebook fully
offline, so we don't run it — but here's the shape:

```python
import os
if os.getenv("OPENAI_API_KEY"):
    spec = uniko.LlmSpec.openai("llm/default", "gpt-4o-mini")        # reads OPENAI_API_KEY
    uni_llm = await uniko.Uniko.builder().in_memory().llm(spec).build()
    ans = await uni_llm.agent("assistant").answer("What stack does Alice use?")
    print(ans.text)
    for c in ans.citations():
        print("  cited:", c.message_id or c.artifact_id)
```

`LlmSpec.mistralrs("local", "<model>")` runs a local model instead — fully offline, no API key.


## 6. Query the graph with Cypher

`agent.query(cypher)` runs a **read-only** Cypher query and returns rows as plain dicts; a returned node is
`{"id", "labels", "properties"}`. Here we pull nodes and aggregate label counts in Python, then peek at the
raw Message node properties.


In [ ]:
from collections import Counter

rows = await agent.query("MATCH (n) RETURN n LIMIT 500")
labels = Counter(label for row in rows for label in row["n"]["labels"])
print("nodes by label:", dict(labels))

In [ ]:
rows = await agent.query("MATCH (m:Message) RETURN m LIMIT 3")
for row in rows:
    print(row["m"]["properties"])

## 7. Retrieve messages and documents by id

`agent.data` fetches typed views by external id (returning `None`, never raising, on a miss). You can also
**ingest standalone documents** with `session.ingest(IngestSource...)` and read their reassembled text back.


In [ ]:
msg = await agent.data.message("m1")
print("message m1:", msg.sender_id, "@", msg.timestamp.isoformat())
print("  content:", msg.content)

outcome = await session.ingest(
    uniko.IngestSource.from_text("# Benchmark Spec\n\nTarget: 1000 recalls/sec.").with_id("spec-1")
)
print("ingested:", outcome.kind, outcome.artifact_id, "| chunks:", len(outcome.chunk_node_ids))

artifact = await agent.data.artifact("spec-1")
print("artifact text:", artifact.text[:80].replace(chr(10), " "))

## 8. Goals and tasks

The working-memory surface. Goals are owned by the agent's participant (which exists because the assistant
already spoke). Create goals/tasks, drive their phases (`planned → active → completed`/`abandoned`), and
expand a goal's working context.


In [ ]:
goals = agent.goals

gid = await goals.create(
    "Ship the recall benchmark",
    goal_id="goal-bench",
    description="1000 recalls/sec by end of June",
    metrics={"target_qps": 1000},
)
await goals.start("goal-bench")
tid = await goals.create_task(
    "Wire up the harness", goal_id="goal-bench", task_id="task-harness", priority=0.9
)
print("goal node:", gid, "| task node:", tid)

g = await goals.get("goal-bench")
print(f"goal: {g.title!r} phase={g.phase} status={g.status!r}")
for t in await goals.tasks_of("goal-bench"):
    print(f"  task: {t.title!r} phase={t.phase} priority={t.priority}")

ctx = await goals.context("goal-bench")
print("context entities:", ctx.entities)

await goals.complete("goal-bench", {"shipped": True})
print("after complete -> phase:", (await goals.get("goal-bench")).phase)
print("active goals now:", [gg.goal_id for gg in await goals.active()])

## 9. Logic & reasoning (Locy)

uniko reasons **inside the engine**, not in an LLM. Three surfaces:

- **Rules** — `define_rule` / `run_rule` register and run a Locy derivation rule.
- **`assume`** — fork the graph, apply hypothetical mutations, query, then roll back automatically.
- **`abduce`** — find the minimal graph additions that would satisfy a conclusion.


In [ ]:
nid = await agent.define_rule(
    "reachable", "CREATE RULE reachable AS MATCH (a:Episode) YIELD KEY a"
)
print("rule node:", nid)
print("run_rule rows:", await agent.run_rule("reachable", ["a"]))

In [ ]:
rows = await (
    agent.assume(
        "ASSUME { CREATE (:Fact {fact_id: 'hyp-1', subject: 'benchmark', "
        "predicate: 'status', object: 'green'}) }"
    )
    .then_query("MATCH (f:Fact {subject: 'benchmark'}) RETURN f")
    .run()
)
print("inside assume, matched facts:", len(rows))

after = await agent.query("MATCH (f:Fact {subject: 'benchmark'}) RETURN f")
print("after assume (rolled back), matched facts:", len(after))

In [ ]:
result = await agent.abduce("ABDUCE reachable WHERE a.kind = 'query'")
print("abduced modifications:", len(result.modifications))
for mod in result.modifications:
    print("  ", mod)

## 10. Scoped recall and query

`Scope` confines reads to a slice of the graph — by session, participant, or time window — and feeds the
`recall_in` / `query_in` variants.


In [ ]:
await agent.session("rockets").observe(uniko.Turn("alice", "alpha notes about rockets").id("r1"))
await agent.session("garden").observe(uniko.Turn("alice", "beta notes about gardening").id("g1"))

scoped = await agent.recall_in("rockets", uniko.Scope().sessions(["rockets"]))
print("recall scoped to 'rockets' session -> items:", len(scoped))

cutoff = datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(hours=1)
rows = await agent.query_in(
    "MATCH (n) RETURN n LIMIT 5",
    uniko.Scope().sessions(["rockets"]).since(cutoff),
)
print("query scoped to 'rockets' since 1h ago -> rows:", len(rows))

## 11. Forgetting and erasure

uniko distinguishes **soft** forgetting (redact content, keep the provenance chain) from **hard** deletion,
and supports GDPR-style participant erasure. Each returns a `DeletionReport`.


In [ ]:
def show(label, r):
    print(f"{label}: redacted={r.nodes_redacted} deleted={r.nodes_deleted} "
          f"edges={r.edges_deleted} facts_invalidated={r.facts_invalidated} root_existed={r.root_existed}")

show("forget_turn(m3)   ", await session.forget_turn("m3"))      # soft: redact
show("delete_turn(m5)   ", await session.delete_turn("m5"))      # hard: one turn
show("delete_session    ", await agent.delete_session("garden")) # hard: whole session
show("forget_participant", await agent.forget_participant("alice"))  # GDPR erasure

## 12. Reset and shut down

`purge()` wipes the graph (handy between experiments). `shutdown()` drains the pipeline and **consumes the
instance** — first drop every `agent` / `session` / `goals` handle (the same rule as the sync path), then
call it. Using `uni` afterward raises.

In [ ]:
report = await uni.purge()
print(f"purge -> {report.nodes_deleted} nodes, {report.edges_deleted} edges")

# shutdown() consumes the instance and requires every Agent/Session/Goals
# handle to be dropped first (Python has no move semantics).
del agent, session, goals
await uni.shutdown()
print("shutdown complete; the handle is now poisoned.")

## 13. The synchronous API (no event loop)

Every verb has a blocking `*_sync` twin for scripts, notebooks without top-level await, and sync handlers.
It blocks on the shared runtime and releases the GIL across the Rust work. One rule: **drop the `agent` /
`session` handles before `shutdown_sync()`** (Python has no move semantics).


In [ ]:
uni2 = uniko.Uniko.in_memory_sync()
agent2 = uni2.agent("assistant")
session2 = agent2.session("c1")

session2.observe_sync(uniko.Turn("alice", "Sync world: I prefer tea over coffee."))
bundle2 = agent2.recall_sync("beverage preference")
print("sync recall items:", len(bundle2))

del session2, agent2          # drop handles first
uni2.shutdown_sync()
print("sync path complete.")

## 14. Recap & next steps

You drove the whole memory loop: **open → observe → recall → query → retrieve → goals/tasks → logic →
scoped recall → forget → shut down**, plus the synchronous twin.

- **Python SDK docs:** https://rustic-ai.github.io/uniko/python/
- **Quickstart:** https://rustic-ai.github.io/uniko/python/quickstart/
- **API reference:** https://rustic-ai.github.io/uniko/python/api/
- **Concepts (memory model, facts & drift, recall cascade):** https://rustic-ai.github.io/uniko/concepts/architecture/
